# Chapitre 2 - Lab : tables, JSONB et requêtes récursives dans PostgreSQL

Ce lab confronte le cours à une vraie base relationnelle qui sait aussi stocker des
documents : PostgreSQL 16 et son type **JSONB** (JSON stocké sous forme binaire
décomposée, indexable). Nous allons :

1. ranger les mêmes CV en tables normalisées et en documents JSONB, et comparer leur
   taille ;
2. lire un profil complet : plusieurs requêtes, une jointure, un document, un agrégat
   JSON calculé par le serveur ;
3. indexer des valeurs **imbriquées** dans les documents (index GIN) pour interroger une
   relation plusieurs-à-plusieurs dans les deux sens ;
4. mesurer le prix de la localité : une petite modification dans un gros document,
   comptée en octets de journal (WAL) ;
5. migrer un schéma : ce qui est instantané et ce qui réécrit la table ;
6. stocker un graphe dans deux tables et l'interroger avec `WITH RECURSIVE`, y compris
   la détection de cycles.

**Démarrer et arrêter les services** (depuis la racine du dépôt) :

```bash
docker compose -f docker/datamodels/compose.yml -p ddia_datamodels up -d     # démarrer
docker compose -f docker/datamodels/compose.yml -p ddia_datamodels down -v   # arrêter et effacer
```

PostgreSQL écoute sur le port **10332** de la machine (utilisateur, mot de passe et base :
`ddia` / `ddia` / `modeles`).

In [1]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg
from psycopg.types.json import Jsonb

from ddia import datamodels as dm
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 120)
fmt = lambda v: f"{v:,.0f}".replace(",", " ")

DSN = "host=localhost port=10332 user=ddia password=ddia dbname=modeles"
con = None
for essai in range(30):
    try:
        con = psycopg.connect(DSN, autocommit=True)
        break
    except psycopg.OperationalError:
        time.sleep(1)
assert con is not None, "PostgreSQL ne répond pas : avez-vous lancé docker compose up ?"
print(con.execute("SELECT version()").fetchone()[0][:60])


def q(sql, params=None):
    return con.execute(sql, params).fetchall()


def plan(sql, params=None):
    return "\n".join(r[0] for r in q("EXPLAIN (ANALYZE, BUFFERS, TIMING OFF, SUMMARY OFF) " + sql, params))


def wal_bytes(fn):
    """Octets de journal (WAL) écrits pendant fn()."""
    a = q("SELECT pg_current_wal_lsn()")[0][0]
    fn()
    b = q("SELECT pg_current_wal_lsn()")[0][0]
    return float(q("SELECT pg_wal_lsn_diff(%s, %s)", (b, a))[0][0])

PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linu
